#Load the Data and Unzip it

In [ ]:
!ls

RCNN_Dataset.zip  sample_data


In [ ]:
!unzip RCNN_Dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: RCNN Dataset/Annotations/eb35b80a2ff3033ed1584d05fb1d4e9fe777ead218ac104497f5c978a4efbcb0_640_jpg.rf.7a0c1ce3f2184b4d762e8a887a1aae04.xml  
  inflating: RCNN Dataset/Annotations/eb35b90620f1023ed1584d05fb1d4e9fe777ead218ac104497f5c978a4efb4bb_640_jpg.rf.f9059d21b8fef3b4694e5bc49ecee70c.xml  
  inflating: RCNN Dataset/Annotations/eb35b9062cf6043ed1584d05fb1d4e9fe777ead218ac104497f5c978a4efbcb0_640_jpg.rf.b6be480bcb7af5dff9a247215bfdc7a0.xml  
  inflating: RCNN Dataset/Annotations/eb35b9062df2093ed1584d05fb1d4e9fe777ead218ac104497f5c978a4efbcb0_640_jpg.rf.458b2850990696d164c87f0ff89807f3.xml  
  inflating: RCNN Dataset/Annotations/eb36b40829f3013ed1584d05fb1d4e9fe777ead218ac104497f5c978a4eebdbd_640_jpg.rf.68f2e5af0bacdb0603f16e7d05fa91db.xml  
  inflating: RCNN Dataset/Annotations/eb36b6082ef3013ed1584d05fb1d4e9fe777ead218ac104497f5c978a4efbcb0_640_jpg.rf.6bee51655848062e81fed2e203b83829.xml  
  inflating: RCNN Dataset/Annot

#Downlode the package

In [ ]:
!pip install selectivesearch opencv-python-headless tqdm

  Preparing metadata (setup.py) ... done
  Created wheel for selectivesearch: filename=selectivesearch-0.4-py3-none-any.whl size=4336 sha256=ba0566f4acf316a609473554af06403dc894fafa81844b3540e6eeebf98448eb
  Stored in directory: /root/.cache/pip/wheels/7f/9b/c7/58b71f1e9fe4aa0ef8affd1c673f8818bc22a5091ea8cbbe93
Successfully built selectivesearch


#Import the Lib

In [ ]:
import os, random, math, joblib
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import selectivesearch
import cv2
from PIL import Image
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from pathlib import Path

#Initilize the Path

In [ ]:

IMAGES_DIR = Path("/content/RCNN Dataset/Images")
ANN_DIR = Path("/content/RCNN Dataset/Annotations")
CLASSES_TXT = Path("/content/RCNN Dataset/classes.txt")

OUT_DIR = Path("/content/rcnn_train_output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
print(IMAGES_DIR.exists())
print(ANN_DIR.exists())
print(CLASSES_TXT.exists())


True
True
True


# model / training params

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS =  100               # increase for better results
MAX_PROPOSALS_PER_IMAGE = 600
MIN_BOX_AREA = 400
POS_IOU = 0.5
NEG_IOU = 0.3
MAX_POS_PER_IMG = 16
MAX_NEG_PER_IMG = 32
STEPS_PER_EPOCH = 200      # generator steps per epoch
VAL_STEPS = 50
LEARNING_RATE = 1e-4
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# load classes

In [ ]:
CLASSES_TXT = r"/content/RCNN Dataset/classes.txt"
with open(CLASSES_TXT) as f:
    class_names = [c.strip() for c in f if c.strip()]
# add background as class 0 (we will map background index 0)
# We choose class indices: 0 = background, 1..N = actual classes
num_classes = len(class_names) + 1
print("Classes:", class_names, "=> num_classes incl background:", num_classes)

Classes: ['elephant', 'leopard', 'wildbore'] => num_classes incl background: 4


# helper: parse VOC xml

In [ ]:
def parse_voc(xml_path):
    root = ET.parse(str(xml_path)).getroot()
    objs = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip().lower()
        b = obj.find("bndbox")
        xmin, ymin, xmax, ymax = [int(float(b.find(t).text)) for t in ("xmin","ymin","xmax","ymax")]
        objs.append({"name": name, "bbox": (xmin,ymin,xmax,ymax)})
    return objs

# IoU

In [ ]:
def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    interW = max(0, xB - xA + 1); interH = max(0, yB - yA + 1)
    inter = interW * interH
    areaA = (boxA[2]-boxA[0]+1)*(boxA[3]-boxA[1]+1)
    areaB = (boxB[2]-boxB[0]+1)*(boxB[3]-boxB[1]+1)
    uni = areaA + areaB - inter
    return inter/uni if uni>0 else 0

# Selective search proposals for an image (RGB)

In [ ]:
def get_selective_search_regions(image, scale=500, sigma=0.9, min_size=10, max_proposals=MAX_PROPOSALS_PER_IMAGE):
    img_lbl, regions = selectivesearch.selective_search(image, scale=scale, sigma=sigma, min_size=min_size)
    candidates = set()
    for r in regions:
        x,y,w,h = r['rect']
        if w==0 or h==0: continue
        if w*h < MIN_BOX_AREA: continue
        candidates.add((x, y, x+w-1, y+h-1))
    candidates = sorted(list(candidates), key=lambda b: (b[2]-b[0])*(b[3]-b[1]), reverse=True)[:max_proposals]
    return candidates

# bbox transform targets (tx,ty,tw,th) relative to proposal

In [ ]:
def bbox_transform(ex_roi, gt_box):
    px = (ex_roi[0] + ex_roi[2]) / 2.0
    py = (ex_roi[1] + ex_roi[3]) / 2.0
    pw = ex_roi[2] - ex_roi[0] + 1.0
    ph = ex_roi[3] - ex_roi[1] + 1.0
    gx = (gt_box[0] + gt_box[2]) / 2.0
    gy = (gt_box[1] + gt_box[3]) / 2.0
    gw = gt_box[2] - gt_box[0] + 1.0
    gh = gt_box[3] - gt_box[1] + 1.0
    tx = (gx - px) / pw
    ty = (gy - py) / ph
    tw = np.log(gw / pw + 1e-8)
    th = np.log(gh / ph + 1e-8)
    return [tx, ty, tw, th]

# clamp and safe crop

In [ ]:
def clamp_box(box, W, H):
    x1,y1,x2,y2 = box
    x1 = max(0, min(int(x1), W-1)); y1 = max(0, min(int(y1), H-1))
    x2 = max(0, min(int(x2), W-1)); y2 = max(0, min(int(y2), H-1))
    if x2 <= x1 or y2 <= y1: return None
    return (x1,y1,x2,y2)

# generator that yields batches of (patch_images, [cls_onehot, bbox_targets])

In [ ]:
# make a lowercased class-name list for fast membership checks
class_names_lc = [c.strip().lower() for c in class_names]

# collect image paths (ensure IMAGES_DIR is a Path object)
image_paths = sorted([p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in (".jpg",".jpeg",".png")])
print("Image count:", len(image_paths))

# keep track of unknown labels we've seen (to warn only once per label)
_unknown_labels_seen = set()

def generator(batch_size=BATCH_SIZE, augment=False):
    while True:
        X_batch = []
        cls_batch = []
        bbox_batch = []
        while len(X_batch) < batch_size:
            img_p = random.choice(image_paths)
            xml_p = ANN_DIR / (img_p.stem + ".xml")
            if not xml_p.exists():
                # skip images without matching xml
                continue

            # parse and filter GT objects to only known class names
            gt_objs_all = parse_voc(xml_p)
            gt_objs = []
            for o in gt_objs_all:
                name = o['name'].strip().lower()
                if name in class_names_lc:
                    gt_objs.append({"name": name, "bbox": o['bbox']})
                else:
                    if name not in _unknown_labels_seen:
                        _unknown_labels_seen.add(name)
                        print(f"Warning: unknown label '{name}' in {xml_p}. Skipping these objects. (Fix classes.txt or XMLs.)")

            gt_boxes = [o['bbox'] for o in gt_objs]
            gt_names = [o['name'] for o in gt_objs]

            img_bgr = cv2.imread(str(img_p))
            if img_bgr is None:
                continue
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            H, W = img_rgb.shape[:2]
            proposals = get_selective_search_regions(img_rgb)

            pos_added = 0
            neg_added = 0
            # shuffle proposals for random sampling
            random.shuffle(proposals)
            for prop in proposals:
                if len(X_batch) >= batch_size:
                    break
                clp = clamp_box(prop, W, H)
                if clp is None:
                    continue
                x1,y1,x2,y2 = clp
                patch = img_rgb[y1:y2+1, x1:x2+1]
                if patch.size == 0:
                    continue

                # compute best IoU among VALID gt_boxes only (if none, best_iou remains 0)
                best_iou = 0.0
                best_idx = -1
                for i, g in enumerate(gt_boxes):
                    val = iou(prop, g)
                    if val > best_iou:
                        best_iou = val
                        best_idx = i

                # Decide positive / negative using best_iou, but only if we have valid GTs
                if best_iou >= POS_IOU and best_idx != -1 and pos_added < MAX_POS_PER_IMG:
                    # positive: class = gt class index + 1 (0 reserved for background)
                    cls_name = gt_names[best_idx]  # already lowercased and validated
                    cls_idx = 1 + class_names_lc.index(cls_name)  # 1..N
                    tx,ty,tw,th = bbox_transform(prop, gt_boxes[best_idx])
                    pos_added += 1
                elif best_iou < NEG_IOU and neg_added < MAX_NEG_PER_IMG:
                    # negative (background)
                    cls_idx = 0
                    tx,ty,tw,th = 0.0,0.0,0.0,0.0
                    neg_added += 1
                else:
                    # either IoU in gray-zone, or too many pos/neg for this image
                    continue

                # prepare image patch
                patch_resized = cv2.resize(patch, IMG_SIZE)
                arr = patch_resized.astype(np.float32)
                arr = preprocess_input(arr)  # VGG preprocess
                X_batch.append(arr)

                # class one-hot
                onehot = np.zeros(num_classes, dtype=np.float32)
                onehot[cls_idx] = 1.0
                cls_batch.append(onehot)
                bbox_batch.append([tx,ty,tw,th])

        X = np.stack(X_batch, axis=0)
        y_cls = np.stack(cls_batch, axis=0)
        y_bbox = np.stack(bbox_batch, axis=0)
        yield X, {"cls_output": y_cls, "bbox_output": y_bbox}


Image count: 2641


# Build model: backbone + heads

In [ ]:
backbone = VGG16(weights="imagenet", include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = backbone.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(1024, activation="relu")(x)
x = layers.Dropout(0.5)(x)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


# classification head

In [ ]:
cls_out = layers.Dense(num_classes, activation="softmax", name="cls_output")(x)

# bbox regression head

In [ ]:
bbox_out = layers.Dense(4, activation="linear", name="bbox_output")(x)

model = Model(inputs=backbone.input, outputs=[cls_out, bbox_out])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv1        │ (None, 224, 224,  │      1,792 │ input_layer[0][0] │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv2        │ (None, 224, 224,  │     36,928 │ block1_conv1[0][… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pool         │ (None, 112, 112,  │          0 │ block1_conv2[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv1        │ (None, 112, 112,  │     73,856 │ block1_pool[0][0] │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv2        │ (None, 112, 112,  │    147,584 │ block2_conv1[0][… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pool         │ (None, 56, 56,    │          0 │ block2_conv2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv1        │ (None, 56, 56,    │    295,168 │ block2_pool[0][0] │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv2        │ (None, 56, 56,    │    590,080 │ block3_conv1[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_conv3        │ (None, 56, 56,    │    590,080 │ block3_conv2[0][… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block3_pool         │ (None, 28, 28,    │          0 │ block3_conv3[0][… │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv1        │ (None, 28, 28,    │  1,180,160 │ block3_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv2        │ (None, 28, 28,    │  2,359,808 │ block4_conv1[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_conv3        │ (None, 28, 28,    │  2,359,808 │ block4_conv2[0][… │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block4_pool         │ (None, 14, 14,    │          0 │ block4_conv3[0][… │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv1        │ (None, 14, 14,    │  2,359,808 │ block4_pool[0][0] │
│ (Conv2D)            │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block5_conv2        │ (None, 14, 14,    │  2,359,808 │ block5_conv1[0][

 Total params: 15,248,200 (58.17 MB)

 Trainable params: 15,248,200 (58.17 MB)

 Non-trainable params: 0 (0.00 B)

# compile: use combined loss (classification + bbox regression)

In [ ]:
losses = {
    "cls_output": "categorical_crossentropy",
    "bbox_output": "mse"
}
loss_weights = {"cls_output": 1.0, "bbox_output": 1.0}
opt = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=opt, loss=losses, loss_weights=loss_weights, metrics={"cls_output":"accuracy"})

# Train

In [ ]:
train_gen = generator(BATCH_SIZE)
val_gen = generator(BATCH_SIZE)  # you can create a smaller val set by modifying generator for validation images
history = model.fit(train_gen, steps_per_epoch=STEPS_PER_EPOCH, epochs=EPOCHS,
                    validation_data=val_gen, validation_steps=VAL_STEPS)

/usr/local/lib/python3.12/dist-packages/skimage/feature/texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


Epoch 1/100
  3/200 ━━━━━━━━━━━━━━━━━━━━ 59s 301ms/step - bbox_output_loss: 3.9661 - cls_output_accuracy: 0.6007 - cls_output_loss: 1.0280 - loss: 4.9940Warning: unknown label 'leapord' in /content/RCNN Dataset/Annotations/elephant_image_0075_png.rf.5e0a4d12c7bf664bbda58477e5205b0e.xml. Skipping these objects. (Fix classes.txt or XMLs.)
  9/200 ━━━━━━━━━━━━━━━━━━━━ 9:52 3s/step - bbox_output_loss: 2.2289 - cls_output_accuracy: 0.7751 - cls_output_loss: 0.7818 - loss: 3.0107Warning: unknown label 'wild bore' in /content/RCNN Dataset/Annotations/wildbore_image_0517_png.rf.029db83f971b78858e0950aeb262d473.xml. Skipping these objects. (Fix classes.txt or XMLs.)
200/200 ━━━━━━━━━━━━━━━━━━━━ 1065s 5s/step - bbox_output_loss: 0.3506 - cls_output_accuracy: 0.9359 - cls_output_loss: 0.3460 - loss: 0.6966 - val_bbox_output_loss: 0.0079 - val_cls_output_accuracy: 0.9513 - val_cls_output_loss: 0.0997 - val_loss: 0.1077
Epoch 2/100
176/200 ━━━━━━━━━━━━━━━━━━━━ 1:49 5s/step - bbox_output_loss: 0.053

# Save the model as HDF5

In [ ]:
out_path = OUT_DIR / "animal_detection_rcnn.h5"
model.save(str(out_path))
print("Saved model to:", out_path)